# NB14f — Per-Model Relative Cluster + Behavioral Validation (ANN-014)

**Datum:** 2026-05-28
**Lock:** [ANN-014](../docs/decisions/ANN-014-per-model-relative-cluster-behavioral-stability.md) — Per-Model Cluster + Behavioral Stability
**Vorgänger:** [NB14e](14e_cluster_premium_calibration.ipynb) (methodischer Bug: Mean-Cutoff auf alle Seeds appliziert)

**Was sich ändert vs NB14e:**
- ❌ Vorher: `LOCKED_PREMIUM_CLUSTER = mean(per_seed_clusters)` → auf alle Seeds appliziert (Seeds 42+7 hatten 0 Trades weil Mean > ihre Maxima)
- ✅ Jetzt: jeder Seed nutzt SEINEN eigenen extrahierten Cluster-Cutoff (per-model relative)
- ❌ Vorher: `cluster_drift < 0.001` als Stability-Threshold (methodisch falsch)
- ✅ Jetzt: Behavioral Stability (signal_freq CV, PF CV, holdout PF mean, cluster_pct std, MDD std)
- ✅ NEU: Per-Symbol Quality-Tiering (Pair-Tiering Vorbereitung für V1.5)

**Goal:** V1-Production-Lock-Entscheidung. Wenn Behavioral Stability PASSED + alle Profile haben Hold-Out PF ≥ 1.3 → V1-Ready.

**Output:** `/results/nb14f/` mit Per-Seed-Multi-Run-Statistik + Pair-Tiering.

## Section 0 — Setup + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14f_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS, TRAIN_END, VAL_END,
    DATA_RAW, HTF_CONTEXT_TIMEFRAMES,
)
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from tqdm.notebook import tqdm
from core.analysis.probability_diagnostic import (
    extract_premium_cluster, apply_cluster_cutoff_mask,
    behavioral_stability_check, pair_level_quality_check,
)
from core.analysis.product_metrics import signals_per_day

TF = '5m'
R_VALUE = 1.5
WIN_R, LOSS_R = R_VALUE, 1.0
SEEDS = [42, 1, 7]
MIN_CLUSTER_SIZE_PCT = 0.5
DECIMAL_PLACES = 4

NY_SESSION_HOUR_START = 13
NY_SESSION_HOUR_END   = 22

PROFILES = {
    'Aggressive':   {'require_htf': False, 'require_ny': False},
    'Balanced':     {'require_htf': True,  'require_ny': False},
    'Conservative': {'require_htf': True,  'require_ny': True},
}

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14f'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Seeds: {SEEDS}  |  TF: {TF}  |  ANN-014 Per-Model-Cluster Mode')
print(f'Output: {OUTPUT_DIR}')

## Section 1 — Load Data + Walk-Forward

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

# --- Auto-Build _extended.parquet for missing symbols (NB14 Section 2 pattern) ---
# Required when new symbols are added (e.g., ANN-015 pool expansion NZDUSD/USDCAD).
# HTF context features for the primary TF read RAW OHLCV files (_1h.parquet, _4h.parquet)
# directly via attach_htf_context — they do NOT need separate _extended files.
if IN_COLAB:
    DATA_RAW_PATH = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
else:
    DATA_RAW_PATH = DATA_RAW
DATA_RAW_PATH.mkdir(parents=True, exist_ok=True)

def load_ohlcv_raw(symbol, tf):
    p = DATA_RAW_PATH / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

def build_extended_features(symbol, tf):
    """Compute extended features for one (symbol, tf) pair, including labels."""
    ohlcv = load_ohlcv_raw(symbol, tf)
    if ohlcv is None or ohlcv.empty:
        return None
    base = compute_features(ohlcv)
    htf_data = {}
    for htf in HTF_CONTEXT_TIMEFRAMES:
        d = load_ohlcv_raw(symbol, htf)
        htf_data[htf] = compute_features(d) if (d is not None and not d.empty) else pd.DataFrame()
    base = attach_htf_context(base, htf_data.get('1h', pd.DataFrame()), htf_data.get('4h', pd.DataFrame()))
    macro_path = DATA_RAW_PATH / 'macro_daily.parquet'
    macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()
    base = attach_macro(base, macro)
    atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
    ema_align = base['ema_alignment'].fillna(0).values if 'ema_alignment' in base.columns else np.zeros(len(base))
    smc = compute_smc_features(ohlcv, atr14, ema_align)
    sess = compute_session_features(ohlcv, atr14)
    inter = compute_htf_interactions(base)
    ext = pd.concat([base, smc, sess, inter], axis=1)
    label_path = DATA_PROCESSED_LOCAL / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
    if not label_path.exists():
        return None    # caller treats this as "labels missing" — fail with clear message
    labels = pd.read_parquet(label_path)
    cols_to_join = ['label']
    if 'hit_bar_offset' in labels.columns:
        cols_to_join.append('hit_bar_offset')
    ext = ext.join(labels[cols_to_join], how='inner')
    if 'hit_bar_offset' not in ext.columns:
        ext['hit_bar_offset'] = 24
    ext['symbol'] = symbol
    ext['timeframe'] = tf
    return ext

needed_symbols = FX_TRAIN_SYMBOLS + FX_HOLDOUT_SYMBOLS
missing_ext = [s for s in needed_symbols
               if not (DATA_EXT / f'{s}_{TF}_extended.parquet').exists()]
if missing_ext:
    print(f'Building {len(missing_ext)} missing _extended files for TF={TF} (raw OHLCV + features + labels)...')
    skipped_labels = []
    for s in tqdm(missing_ext, desc='Build extended'):
        ext = build_extended_features(s, TF)
        if ext is None:
            skipped_labels.append(s)
            continue
        out = DATA_EXT / f'{s}_{TF}_extended.parquet'
        ext.to_parquet(out, compression='zstd')
    if skipped_labels:
        print(f'\n  {len(skipped_labels)} symbol(s) hatten KEINE Labels:')
        for s in skipped_labels:
            print(f'    {s} {TF} — labels_{s}_{TF}_R{R_VALUE}.parquet fehlt')
        raise SystemExit('Labels fehlen — NB04 (Triple-Barrier-Labeling) ausführen, dann NB14f neu starten.')
    if IN_COLAB:
        print('Syncing extended features to Drive backup...')
        subprocess.run(['rsync', '-ah', f'{DATA_EXT}/', f'{EXT_DRIVE_BACKUP}/'])

missing = [s for s in needed_symbols
           if load_ext(s, TF) is None or load_ext(s, TF).empty]
if missing:
    raise SystemExit(f'Missing _extended for: {missing} on primary TF {TF} — Build failed unexpectedly.')

frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    d['symbol'] = s
    frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'Features: {len(FEATURE_COLS)}')

pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val   = val_df[FEATURE_COLS].values.astype(np.float32)
y_val   = binary_label_for_long(val_df['label']).values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
del pool, pool_c
gc.collect()

## Section 2 — Training Helper + Filter-Apply

In [ ]:
BASE_PARAMS = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 100,
    'lambda_l2': 1.0, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'is_unbalance': True,
    'verbose': -1, 'n_jobs': -1,
}

def train_with_seed(X_tr, y_tr, X_vl, y_vl, seed):
    params = dict(BASE_PARAMS)
    params['seed'] = seed
    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    return lgb.train(params, td, valid_sets=[td, vd], valid_names=['train','val'],
                      callbacks=[lgb.early_stopping(10, verbose=False), lgb.log_evaluation(period=0)])

def apply_filters_per_seed(df, proba, profile_name, cluster_value):
    """Filter mask. ANN-014: cluster_value is SEED-SPECIFIC, not global."""
    cfg_prof = PROFILES[profile_name]
    rounded = np.round(proba, DECIMAL_PLACES)
    mask = rounded >= cluster_value
    if cfg_prof['require_htf']:
        mask = mask & (df['htf_ltf_agree_bull'].values == 1)
    if cfg_prof['require_ny']:
        hour_utc = df.index.hour.values
        mask = mask & ((hour_utc >= NY_SESSION_HOUR_START) & (hour_utc < NY_SESSION_HOUR_END))
    return mask

def metrics_for_mask(labels_triple, mask, n_bars, n_symbols, durations=None):
    n_sig = int(mask.sum())
    if n_sig == 0:
        return {'n_trades': 0, 'wins': 0, 'losses': 0, 'pf': 0.0, 'wr': 0.0,
                'sigs_per_day_per_symbol': 0.0, 'mdd': 0.0, 'avg_duration': 0.0}
    sel = mask.values if hasattr(mask, 'values') else mask
    labs = labels_triple[sel]
    wins = int((labs == 1).sum())
    losses = int((labs == -1).sum())
    pf = (wins * WIN_R) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    pnl = np.where(labs == 1, WIN_R, np.where(labs == -1, -1.0, 0.0))
    equity = np.cumsum(pnl); rmax = np.maximum.accumulate(equity)
    mdd = float((rmax - equity).max()) if len(equity) > 0 else 0.0
    sigs_pd = signals_per_day(n_sig, n_bars, TF, n_symbols)
    avg_dur = float(durations[sel].mean()) if (durations is not None and n_sig > 0) else 0.0
    return {'n_trades': n_sig, 'wins': wins, 'losses': losses,
            'pf': pf, 'wr': wr, 'sigs_per_day_per_symbol': sigs_pd,
            'mdd': mdd, 'avg_duration': avg_dur}

print('Helpers defined.')

## Section 3 — Per-Seed Training + Per-Seed Cluster-Extraction

**Wichtig (ANN-014):** Jeder Seed trainiert sein eigenes Modell UND nutzt seinen eigenen extrahierten Cluster-Cutoff.

In [ ]:
per_seed = {}  # seed -> {model, proba_val, proba_test, cluster}

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    model = train_with_seed(X_train, y_train, X_val, y_val, seed)
    pv = model.predict(X_val)
    pt = model.predict(X_test)
    cluster = extract_premium_cluster(pv, decimal_places=DECIMAL_PLACES,
                                       min_cluster_size_pct=MIN_CLUSTER_SIZE_PCT)
    per_seed[seed] = {
        'model':      model,
        'proba_val':  pv,
        'proba_test': pt,
        'cluster':    cluster,
    }
    if cluster['success']:
        print(f'VAL proba range: [{pv.min():.4f}, {pv.max():.4f}]')
        print(f'TEST proba range: [{pt.min():.4f}, {pt.max():.4f}]')
        print(f'OWN Premium-Cluster: value={cluster["premium_cluster_value"]:.4f}  '
              f'pct={cluster["premium_cluster_pct"]:.2f}%  '
              f'size={cluster["premium_cluster_size"]}')
    else:
        print(f'CLUSTER EXTRACTION FAILED: {cluster.get("error")}')

## Section 4 — Per-Seed Inferenz mit Eigenem Cluster

Jeder Seed evaluiert mit SEINEM eigenen Cluster (Fix vs NB14e).

In [ ]:
test_labels_triple = test_df['label'].values
test_durations = test_df['hit_bar_offset'].values if 'hit_bar_offset' in test_df.columns else None

in_sample_per_seed_profile = []
for seed in SEEDS:
    data = per_seed[seed]
    if not data['cluster']['success']:
        continue
    seed_cluster_value = data['cluster']['premium_cluster_value']
    for profile in PROFILES.keys():
        mask = apply_filters_per_seed(test_df, data['proba_test'], profile, seed_cluster_value)
        m = metrics_for_mask(test_labels_triple, mask, len(test_df), len(FX_TRAIN_SYMBOLS), test_durations)
        m['seed']    = seed
        m['profile'] = profile
        m['cluster_value'] = seed_cluster_value
        in_sample_per_seed_profile.append(m)

in_sample_df = pd.DataFrame(in_sample_per_seed_profile)
print('=== IN-SAMPLE TEST — per seed × profile (PER-SEED CLUSTER USED) ===')
cols = ['seed', 'profile', 'cluster_value', 'n_trades', 'wr', 'pf', 'sigs_per_day_per_symbol', 'mdd', 'avg_duration']
print(in_sample_df[cols].round(4).to_string(index=False))

## Section 5 — Hold-Out Validation per Seed (Per-Symbol)

In [ ]:
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')

holdout_per_seed_sym_prof = []
for seed in SEEDS:
    data = per_seed[seed]
    if not data['cluster']['success']:
        continue
    seed_cluster_value = data['cluster']['premium_cluster_value']
    model = data['model']

    for sym in FX_HOLDOUT_SYMBOLS:
        h = load_ext(sym, TF)
        h = h.dropna(subset=FEATURE_COLS + ['label'])
        h = h[h.index >= val_end_ts]
        if h.empty:
            continue
        proba_h = model.predict(h[FEATURE_COLS].values.astype(np.float32))
        labels_h = h['label'].values
        durations_h = h['hit_bar_offset'].values if 'hit_bar_offset' in h.columns else None
        for profile in PROFILES.keys():
            mask = apply_filters_per_seed(h, proba_h, profile, seed_cluster_value)
            m = metrics_for_mask(labels_h, mask, len(h), 1, durations_h)
            m['seed'] = seed; m['profile'] = profile; m['symbol'] = sym
            holdout_per_seed_sym_prof.append(m)
        del h, proba_h
        gc.collect()

holdout_df = pd.DataFrame(holdout_per_seed_sym_prof)
print('=== HOLD-OUT per seed × symbol × profile ===')
print(holdout_df[['seed', 'symbol', 'profile', 'n_trades', 'wr', 'pf', 'sigs_per_day_per_symbol']].round(4).to_string(index=False))

## Section 6 — Per-Symbol Aggregation (über Seeds, für Pair-Tiering)

In [ ]:
# Per-Symbol pro Profil aggregiert über Seeds (gewichteter PF)
pair_summary = []
for sym in FX_HOLDOUT_SYMBOLS:
    for profile in PROFILES.keys():
        sub = holdout_df[(holdout_df['symbol'] == sym) & (holdout_df['profile'] == profile)]
        if sub.empty:
            continue
        w_wins = int(sub['wins'].sum())
        w_losses = int(sub['losses'].sum())
        w_trades = int(sub['n_trades'].sum())
        w_pf = (w_wins * WIN_R) / w_losses if w_losses > 0 else (float('inf') if w_wins > 0 else 0.0)
        w_wr = w_wins / (w_wins + w_losses) if (w_wins + w_losses) > 0 else 0.0
        pair_summary.append({
            'symbol':       sym,
            'profile':      profile,
            'pf':           round(w_pf, 4) if not np.isinf(w_pf) else float('inf'),
            'wr':           round(w_wr, 4),
            'n_trades':     w_trades,
            'avg_sigs_per_day': round(sub['sigs_per_day_per_symbol'].mean(), 4),
        })
pair_df = pd.DataFrame(pair_summary)
print('=== PAIR-LEVEL Hold-Out (über Seeds aggregiert) ===')
print(pair_df.to_string(index=False))

# Pair-Tiering pro Profil
print('\n=== PAIR-TIERING (ANN-014, threshold supported >= 1.5, experimental >= 1.0) ===')
for profile in PROFILES.keys():
    sub = pair_df[pair_df['profile'] == profile]
    if sub.empty: continue
    tier_input = sub[['symbol', 'pf', 'wr', 'n_trades']].to_dict(orient='records')
    tier_result = pair_level_quality_check(tier_input, min_pf_supported=1.5, min_pf_experimental=1.0)
    print(f'\nProfile: {profile}')
    print(f'  Supported:    {[s["symbol"] for s in tier_result["tiers"].get("supported", [])]}')
    print(f'  Experimental: {[s["symbol"] for s in tier_result["tiers"].get("experimental", [])]}')
    print(f'  Unsupported:  {[s["symbol"] for s in tier_result["tiers"].get("unsupported", [])]}')

## Section 7 — Behavioral Stability Check (ANN-014)

In [ ]:
# Pro Profil: collect per-seed behaviors
behavioral_per_profile = {}
for profile in PROFILES.keys():
    per_seed_behaviors = []
    for seed in SEEDS:
        is_row = in_sample_df[(in_sample_df['seed'] == seed) & (in_sample_df['profile'] == profile)]
        if is_row.empty: continue
        is_row = is_row.iloc[0]
        # Hold-Out PF (gewichtet über symbols für diesen seed × profile)
        ho_sub = holdout_df[(holdout_df['seed'] == seed) & (holdout_df['profile'] == profile)]
        if ho_sub.empty:
            ho_pf = 0.0
        else:
            w_wins = ho_sub['wins'].sum(); w_losses = ho_sub['losses'].sum()
            ho_pf = (w_wins * WIN_R) / w_losses if w_losses > 0 else (float('inf') if w_wins > 0 else 0.0)
        cluster_pct = per_seed[seed]['cluster'].get('premium_cluster_pct', 0.0)
        per_seed_behaviors.append({
            'seed':         seed,
            'sigs_per_day': float(is_row['sigs_per_day_per_symbol']),
            'is_pf':        float(is_row['pf']) if not np.isinf(is_row['pf']) else None,
            'holdout_pf':   float(ho_pf) if not np.isinf(ho_pf) else None,
            'cluster_pct':  float(cluster_pct),
            'mdd':          float(is_row['mdd']),
        })
    stability = behavioral_stability_check(per_seed_behaviors)
    behavioral_per_profile[profile] = {
        'per_seed_behaviors': per_seed_behaviors,
        'stability':          stability,
    }

print('=== BEHAVIORAL STABILITY per Profile (ANN-014) ===')
for profile, data in behavioral_per_profile.items():
    st = data['stability']
    print(f'\nProfile: {profile}')
    print(f'  all_passed: {st["all_passed"]}')
    for metric, info in st['per_metric_status'].items():
        status = 'OK' if info['passed'] else 'FAIL'
        val = info['value']
        val_str = f'{val:.4f}' if isinstance(val, float) else str(val)
        print(f'  [{status}] {metric:30s} value={val_str}  threshold={info["threshold"]}')
    if st['critical_failures']:
        print(f'  Failures: {st["critical_failures"]}')

## Section 8 — Best-Behavior Seed Selection + Pine-Export-Ready Output

In [ ]:
# Welcher Seed produziert das beste Hold-Out-Verhalten (Aggressive Profile)?
agg_per_seed_score = []
for seed in SEEDS:
    sub = holdout_df[(holdout_df['seed'] == seed) & (holdout_df['profile'] == 'Aggressive')]
    if sub.empty: continue
    w_wins = sub['wins'].sum(); w_losses = sub['losses'].sum()
    pf = (w_wins * WIN_R) / w_losses if w_losses > 0 else 0.0
    total_trades = sub['n_trades'].sum()
    agg_per_seed_score.append({
        'seed':          seed,
        'aggressive_holdout_pf': pf,
        'aggressive_total_trades': int(total_trades),
        'cluster_value': per_seed[seed]['cluster'].get('premium_cluster_value'),
    })

# Best seed: höchster Hold-Out-PF (mit mindestens 30 Trades für statistische Power)
candidates = [s for s in agg_per_seed_score if s['aggressive_total_trades'] >= 30]
if candidates:
    best_seed_entry = max(candidates, key=lambda x: x['aggressive_holdout_pf'])
else:
    best_seed_entry = max(agg_per_seed_score, key=lambda x: x['aggressive_total_trades'])
BEST_SEED = best_seed_entry['seed']
BEST_CLUSTER = best_seed_entry['cluster_value']

print('=== Best-Behavior Seed Selection ===')
for s in agg_per_seed_score:
    marker = ' ← SELECTED' if s['seed'] == BEST_SEED else ''
    print(f'  seed={s["seed"]}  cluster={s["cluster_value"]:.4f}  '
          f'holdout_PF={s["aggressive_holdout_pf"]:.2f}  '
          f'trades={s["aggressive_total_trades"]}{marker}')

print(f'\n=== V1-PRODUCTION-READY ===')
print(f'BEST_SEED:    {BEST_SEED}')
print(f'BEST_CLUSTER: {BEST_CLUSTER:.4f}')
print(f'\nDas ist der Wert der in Pine-Codegen hartkodiert wird:')
print(f'  PREMIUM_CLUSTER_VALUE = {BEST_CLUSTER:.4f}')

## Section 9 — Verdict + Pine-Export-Snapshot

In [ ]:
# Best-seed profile statistics
best_seed_profiles = []
for profile in PROFILES.keys():
    is_row = in_sample_df[(in_sample_df['seed'] == BEST_SEED) & (in_sample_df['profile'] == profile)]
    ho_sub = holdout_df[(holdout_df['seed'] == BEST_SEED) & (holdout_df['profile'] == profile)]
    if is_row.empty or ho_sub.empty: continue
    is_row = is_row.iloc[0]
    w_wins = ho_sub['wins'].sum(); w_losses = ho_sub['losses'].sum()
    ho_pf = (w_wins * WIN_R) / w_losses if w_losses > 0 else 0.0
    ho_wr = w_wins / (w_wins + w_losses) if (w_wins + w_losses) > 0 else 0.0
    best_seed_profiles.append({
        'profile':      profile,
        'is_pf':        float(is_row['pf']),
        'is_sigs_day':  float(is_row['sigs_per_day_per_symbol']),
        'is_mdd':       float(is_row['mdd']),
        'ho_pf':        float(ho_pf),
        'ho_wr':        float(ho_wr),
        'ho_n_trades':  int(ho_sub['n_trades'].sum()),
    })
best_seed_df = pd.DataFrame(best_seed_profiles)

verdict = {
    'lock_basis':                'ANN-014 Per-Model Relative Cluster + Behavioral Stability',
    'production_seed':           BEST_SEED,
    'production_cluster_value':  BEST_CLUSTER,
    'profiles_best_seed':        best_seed_profiles,
    'all_profiles_behavioral_stable': all(
        behavioral_per_profile[p]['stability']['all_passed']
        for p in PROFILES.keys() if p in behavioral_per_profile
    ),
    'pair_summary':              pair_summary,
    'seeds_tested':              SEEDS,
}

print('=' * 70)
print(f'NB14f VERDICT — {RUN_DATE}')
print('=' * 70)
print(f'\nProduction Seed: {BEST_SEED}')
print(f'Production Cluster Value: {BEST_CLUSTER:.4f}')
print(f'\nProfile-Quality (Best Seed):')
for p in best_seed_profiles:
    print(f'  {p["profile"]:14s}  IS_PF={p["is_pf"]:.2f}  HO_PF={p["ho_pf"]:.2f}  '
          f'Sigs={p["is_sigs_day"]:.2f}/Tag  HO_Trades={p["ho_n_trades"]}')
print(f'\nBehavioral Stable (all profiles): {verdict["all_profiles_behavioral_stable"]}')
print('=' * 70)

## Section 10 — Persistence

In [ ]:
in_sample_df.to_csv(OUTPUT_DIR / 'metrics' / f'in_sample_per_seed_profile_{RUN_DATE}.csv', index=False)
holdout_df.to_csv(OUTPUT_DIR / 'metrics' / f'holdout_per_seed_symbol_profile_{RUN_DATE}.csv', index=False)
pair_df.to_csv(OUTPUT_DIR / 'metrics' / f'pair_aggregated_{RUN_DATE}.csv', index=False)
best_seed_df.to_csv(OUTPUT_DIR / 'metrics' / f'best_seed_profiles_{RUN_DATE}.csv', index=False)

# Behavioral Stability als JSON
behavioral_serializable = {}
for p, data in behavioral_per_profile.items():
    behavioral_serializable[p] = {
        'per_seed_behaviors': data['per_seed_behaviors'],
        'stability': data['stability'],
    }

snapshot = {
    'experiment_id':           EXPERIMENT_ID,
    'run_date':                RUN_DATE,
    'git_commit':              GIT_COMMIT,
    'seeds':                   SEEDS,
    'verdict':                 verdict,
    'per_seed_clusters':       {str(s): per_seed[s]['cluster'] for s in SEEDS},
    'in_sample_per_seed':      in_sample_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'holdout_per_seed':        holdout_df.replace([np.inf, -np.inf], 'inf').to_dict(orient='records'),
    'pair_summary':            pair_summary,
    'best_seed_profiles':      best_seed_profiles,
    'behavioral_per_profile':  behavioral_serializable,
}
snap_path = OUTPUT_DIR / 'summaries' / f'nb14f_full_snapshot_{RUN_DATE}.json'
with open(snap_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'Snapshot → {snap_path}')

cfg_snap = {
    'experiment_id':         EXPERIMENT_ID,
    'run_date':              RUN_DATE,
    'git_commit':            GIT_COMMIT,
    'base_params':           BASE_PARAMS,
    'seeds':                 SEEDS,
    'min_cluster_size_pct':  MIN_CLUSTER_SIZE_PCT,
    'profiles':              PROFILES,
    'lock_basis':            'ANN-014',
}
cfg_path = OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json'
with open(cfg_path, 'w') as f:
    json.dump(cfg_snap, f, indent=2, default=str)
print(f'Config → {cfg_path}')

## Section 11 — Auto-Push to GitHub

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR cannot read GITHUB_TOKEN: {e}')

    if GH_TOKEN:
        NB_TAG = 'nb14f'
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        # Pull --rebase VOR Copy
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet',
                        'origin', 'main'], check=True)
        print('Pulled latest from origin/main')

        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file(): continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files')

        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                    capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = f'{NB_TAG.upper()} per-model behavioral validation {RUN_DATE} ({len(copied)} files)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                   capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'PUSHED as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)